# Embeddings and Vector Stores in LangChain: OpenAI Embeddings, FAISS, Chroma, and Similarity Search

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/langchain_5)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q langchain langchain-openai langchain-community chromadb

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed a single string
vector = embeddings.embed_query("What is gradient descent?")
print(f"Embedding dimension: {len(vector)}")
print(f"First 5 values: {[round(v, 4) for v in vector[:5]]}")
print(f"Vector norm: {sum(v**2 for v in vector)**0.5:.4f}")

In [ ]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "How do neural networks learn?",
    "What is gradient descent optimization?",
    "Neural networks minimize loss using backpropagation.",
    "The capital of France is Paris.",
    "Transformers use attention mechanisms.",
]

vectors = embeddings.embed_documents(sentences)

query = vectors[0]  # "How do neural networks learn?"
print(f"Query: '{sentences[0]}'\n")
for i, (sent, vec) in enumerate(zip(sentences[1:], vectors[1:]), 1):
    sim = cosine_similarity(query, vec)
    print(f"  [{sim:.4f}] {sent}")

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Build a vector store from documents
docs = [
    Document(page_content="Gradient descent is an optimization algorithm that minimizes a function by iteratively moving in the direction of steepest descent.", metadata={"topic": "optimization"}),
    Document(page_content="Backpropagation computes gradients by applying the chain rule of calculus backwards through the computation graph.", metadata={"topic": "training"}),
    Document(page_content="A convolutional neural network applies learned filters across spatial dimensions to detect features like edges and textures.", metadata={"topic": "architectures"}),
    Document(page_content="The attention mechanism weighs the relevance of each token in the input when computing the representation of each output token.", metadata={"topic": "transformers"}),
    Document(page_content="LSTM networks use gating mechanisms to control what information is stored and forgotten across long sequences.", metadata={"topic": "rnns"}),
    Document(page_content="Transfer learning reuses representations learned on a large dataset as a starting point for a new, smaller dataset.", metadata={"topic": "training"}),
]

# Create vector store — embeds all docs and builds FAISS index
vectorstore = FAISS.from_documents(docs, embeddings)
print(f"Documents indexed: {vectorstore.index.ntotal}")

In [ ]:
# Search for semantically similar documents
query = "How does a model learn to minimize error?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: '{query}'\n")
for i, doc in enumerate(results):
    print(f"Result {i+1} [{doc.metadata['topic']}]:")
    print(f"  {doc.page_content[:100]}...")
    print()

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# (Rebuild vectorstore from above docs...)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)

results_with_scores = vectorstore.similarity_search_with_score(
    "sequence modeling for text", k=3
)

for doc, score in results_with_scores:
    print(f"[L2 distance={score:.4f}] [{doc.metadata['topic']}] {doc.page_content[:80]}...")

In [ ]:
import tempfile
import os
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# vectorstore from above

with tempfile.TemporaryDirectory() as tmpdir:
    index_path = os.path.join(tmpdir, "faiss_index")

    # Save
    vectorstore.save_local(index_path)
    print(f"Saved to: {index_path}")
    print(f"Files: {os.listdir(index_path)}")

    # Load
    loaded_vs = FAISS.load_local(
        index_path, embeddings, allow_dangerous_deserialization=True
    )
    print(f"Loaded index: {loaded_vs.index.ntotal} documents")

    # Verify same results
    result = loaded_vs.similarity_search("gradient descent", k=1)
    print(f"Query result after reload: {result[0].metadata['topic']}")

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
import tempfile

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

with tempfile.TemporaryDirectory() as persist_dir:
    vectorstore = Chroma.from_documents(
        docs,
        embeddings,
        persist_directory=persist_dir,
        collection_name="ml_concepts",
    )

    print(f"Collection size: {vectorstore._collection.count()}")

    # Metadata filtering: only retrieve from 'training' topic
    results = vectorstore.similarity_search(
        "how does a model update its parameters",
        k=2,
        filter={"topic": "training"},
    )

    print(f"\nFiltered to 'training' topic:")
    for doc in results:
        print(f"  [{doc.metadata['topic']}] {doc.page_content[:80]}...")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)

# Configure retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",  # or "mmr" for diversity
    search_kwargs={"k": 2},
)

# Test retriever directly
retrieved = retriever.invoke("attention in transformers")
print(f"Retrieved {len(retrieved)} docs")
for doc in retrieved:
    print(f"  [{doc.metadata['topic']}] {doc.page_content[:60]}...")

# Wire into a RAG chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on this context:\n\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

answer = rag_chain.invoke("How does attention work in neural networks?")
print(f"\nRAG answer: {answer}")

In [ ]:
# WRONG: different models at index vs query time
from langchain_openai import OpenAIEmbeddings

index_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
query_embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# Indices built with index_embeddings cannot be queried with query_embeddings
# The vectors live in different geometric spaces
print("Always use the SAME embedding model for indexing and querying.")
print("Document this as a deployment constraint — it cannot be changed without re-indexing.")